# ColonoMind: Super Agent Unified Evaluation Notebook
**Scenario:** Intra Domain
**Train on:** NTUH
**Test on:** NTUH

This notebook runs the full evaluation pipeline for **5 comparison models** against the ColonoMind Mod-SE(2) Super Agent approach.
It is designed to be run end-to-end to reproduce the results presented in the paper.

## Models Evaluated:
1. ResNet-50
2. DenseNet-121
3. EfficientNet-B4
4. ConvNeXt-Tiny
5. ViT-B/16 (Vision Transformer)


## Section 1: Library Imports and Setup


In [ ]:
import os
import cv2
import numpy as np
import pywt
import scipy.stats
from skimage.feature import graycomatrix, graycoprops
import umap
import itertools
import tqdm
import json
import joblib
import pandas as pd
import matplotlib.pyplot as plt
from hashlib import sha1

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.tree import DecisionTreeClassifier, _tree
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc, cohen_kappa_score
)
from imblearn.over_sampling import SMOTE
import lightgbm as lgb

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D, BatchNormalization, Dropout, Concatenate, Lambda
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.utils.class_weight import compute_class_weight

# Limit GPU memory growth
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)



## Section 1.5: Dataset Fetching & Google Drive Integration


In [ ]:
import os
import sys

# 1. Mount Google Drive for Dataset 1 & 2 (NTUH) if running on Google Colab
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted.")



## Section 2: Dataset Configuration, Feature Extraction & Shared Utilities


In [ ]:
# --- 1. DATASET CONFIGURATION ---
SCENARIO_TYPE = '{scenario_type}'
TRAIN_DATASET = '{train_dataset}'
TEST_DATASET  = '{test_dataset}'

def get_dataset_paths(dataset_name):
    if dataset_name == 'NTUH':
        return ['../Dataset+Code/MES classification_20250313', '../Dataset+Code/MES classification_20250724'], '../Dataset_patient_split/Combined_Dataset_patient_split.csv'
    elif dataset_name == 'LIMUC':
        return ['../Dataset/LIMUC/train_and_validation_sets', '../Dataset/LIMUC/test_set'], '../Dataset_patient_split/LIMUC_official_patient_split.csv'
    elif dataset_name == 'TMC-UCM':
        return ['../Dataset/TMC-UCM/images'], '../Dataset_patient_split/TMC-UCM_official_patient_split.csv'
    else:
        raise ValueError("Invalid Dataset Choice")

TRAIN_DIR, SPLIT_CSV_TRAIN = get_dataset_paths(TRAIN_DATASET)
TEST_DIR, SPLIT_CSV_TEST   = get_dataset_paths(TEST_DATASET)

print(f"✅ Configured for {SCENARIO_TYPE} Domain: Train on {TRAIN_DATASET}, Test on {TEST_DATASET}")


import sys

# Define base save directory directly to Google Drive if available
if 'google.colab' in sys.modules:
    drive_root = '/content/drive/MyDrive/Colonomind_Results'
else:
    drive_root = './Colonomind_Results'

if SCENARIO_TYPE == 'Intra':
    BASE_SAVE_DIR = f"{drive_root}/{SCENARIO_TYPE}_{TRAIN_DATASET}"
else:
    BASE_SAVE_DIR = f"{drive_root}/{SCENARIO_TYPE}_{TRAIN_DATASET}_to_{TEST_DATASET}"

os.makedirs(BASE_SAVE_DIR, exist_ok=True)
print(f"📁 All results will be saved to: {{BASE_SAVE_DIR}}")

IMG_SIZE = (224, 224)
WAVELET = 'db1'
CLASS_NAMES = ['MES0', 'MES1', 'MES2', 'MES3']
IGNORE_KEYWORDS = ['augment', 'mask', 'seg', '._', 'crop']
all_results = {}



In [ ]:
# --- 2. HANDCRAFTED FEATURE EXTRACTORS ---
def extract_wavelet_stats(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    coeffs2 = pywt.dwt2(gray, WAVELET)
    LL, (LH, HL, HH) = coeffs2
    def stats(subband):
        return [
            np.mean(subband), np.std(subband), np.var(subband),
            scipy.stats.entropy(np.abs(subband.flatten()) + 1e-6)
        ]
    hh_energy = np.sum(np.square(HH)) / HH.size
    return stats(LL) + stats(LH) + stats(HL) + stats(HH) + [hh_energy]

def extract_glcm_features(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    distances = [1, 3, 5]
    angles = [0, np.pi/4, np.pi/2, 3*np.pi/4]
    glcm = graycomatrix(gray, distances=distances, angles=angles, levels=256, symmetric=True, normed=True)
    return [
        np.mean(graycoprops(glcm, 'contrast')),
        np.mean(graycoprops(glcm, 'dissimilarity')),
        np.mean(graycoprops(glcm, 'homogeneity'))
    ]

def extract_combined_features(image):
    return extract_wavelet_stats(image) + extract_glcm_features(image)

def extract_patient_id(filename, dataset_name):
    if dataset_name == 'NTUH':
        return filename.split('_')[0]
    elif dataset_name == 'TMC-UCM':
        parts = filename.split('_')
        if len(parts) >= 3 and parts[0] == 'TMC' and parts[1] == 'P':
            return f"{parts[0]}_{parts[1]}_{parts[2]}"
    elif dataset_name == 'LIMUC':
        parts = filename.split('_')
        if len(parts) >= 3 and parts[0] == 'UC' and parts[1] == 'patient':
            return f"{parts[0]}_{parts[1]}_{parts[2]}"
    return None

def load_patient_split(csv_path):
    if not os.path.exists(csv_path): return {}
    df = pd.read_csv(csv_path)
    return dict(zip(df['Patient_ID'].astype(str), df['Split']))

def load_and_split_dataset(dataset_dir, split_csv_path, dataset_name, force_split=None):
    patient_splits = load_patient_split(split_csv_path)
    X_train_img, X_train_feat, y_train, paths_train = [], [], [], []
    X_test_img, X_test_feat, y_test, paths_test = [], [], [], []
    
    for cls in CLASS_NAMES:
        cls_dir = os.path.join(dataset_dir, cls)
        if not os.path.exists(cls_dir): continue
        for img_name in os.listdir(cls_dir):
            if any(k in img_name.lower() for k in IGNORE_KEYWORDS): continue
            
            # Determine split
            if force_split:
                target_split = force_split
            else:
                pid = extract_patient_id(img_name, dataset_name)
                target_split = patient_splits.get(pid, 'Train/Validation') if pid else 'Train/Validation'
                if 'test_set' in dataset_dir.lower():
                    target_split = 'Test'
            
            img_path = os.path.join(cls_dir, img_name)
            img = cv2.imread(img_path)
            if img is None: continue
            img = cv2.resize(img, IMG_SIZE)
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            if target_split == 'Test' or target_split == 'Test Set':
                X_test_img.append(img_rgb)
                X_test_feat.append(extract_combined_features(img_rgb))
                y_test.append(cls)
                paths_test.append(img_path)
            else:
                X_train_img.append(img_rgb)
                X_train_feat.append(extract_combined_features(img_rgb))
                y_train.append(cls)
                paths_train.append(img_path)
                
    return (X_train_img, X_test_img, X_train_feat, X_test_feat, y_train, y_test, paths_train, paths_test)



In [ ]:
# --- 3. LOAD DATA & PREPROCESS ---
print(f"Loading Data...")
X_train_img, X_test_img, X_train_feat, X_test_feat, y_train_label, y_test_label, paths_train, paths_test = [], [], [], [], [], [], [], []

if SCENARIO_TYPE == 'Intra':
    # Intra-domain: use patient splits from the same dataset
    for d_path in TRAIN_DIR:
        tr_i, te_i, tr_f, te_f, y_tr, y_te, p_tr, p_te = load_and_split_dataset(d_path, SPLIT_CSV_TRAIN, TRAIN_DATASET)
        X_train_img.extend(tr_i)
        X_test_img.extend(te_i)
        X_train_feat.extend(tr_f)
        X_test_feat.extend(te_f)
        y_train_label.extend(y_tr)
        y_test_label.extend(y_te)
        paths_train.extend(p_tr)
        paths_test.extend(p_te)
else:
    # Multi-domain: Force 100% Train for Source, 100% Test for Target
    # First, load source dataset as entirely Train
    for d_path in TRAIN_DIR:
        tr_i, _, tr_f, _, y_tr, _, p_tr, _ = load_and_split_dataset(d_path, SPLIT_CSV_TRAIN, TRAIN_DATASET, force_split='Train')
        X_train_img.extend(tr_i)
        X_train_feat.extend(tr_f)
        y_train_label.extend(y_tr)
        paths_train.extend(p_tr)
        
    # Then load target dataset as entirely Test
    for d_path in TEST_DIR:
        _, te_i, _, te_f, _, y_te, _, p_te = load_and_split_dataset(d_path, SPLIT_CSV_TEST, TEST_DATASET, force_split='Test')
        X_test_img.extend(te_i)
        X_test_feat.extend(te_f)
        y_test_label.extend(y_te)
        paths_test.extend(p_te)

X_img_train_raw = np.array(X_train_img)
X_img_test_raw = np.array(X_test_img)
X_feat_train_raw = np.array(X_train_feat)
X_feat_test_raw = np.array(X_test_feat)
img_paths_train = paths_train
img_paths_test = paths_test

print(f"Training samples ({TRAIN_DATASET}): {len(X_img_train_raw)}")
print(f"Testing samples ({TEST_DATASET}): {len(X_img_test_raw)}")

# Normalize images to [0,1]
X_img_train = X_img_train_raw.astype(np.float32) / 255.0
X_img_test  = X_img_test_raw.astype(np.float32) / 255.0

# Encode labels
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train_label)
y_test_encoded  = le.transform(y_test_label)
y_train_cat = to_categorical(y_train_encoded, num_classes=len(le.classes_))
y_test_cat  = to_categorical(y_test_encoded,  num_classes=len(le.classes_))

# Scale Handcrafted Features
scaler = StandardScaler()
X_feat_train_scaled = scaler.fit_transform(X_feat_train_raw)
X_feat_test_scaled  = scaler.transform(X_feat_test_raw)

# --- SMOTE BALANCING ---
print("Applying SMOTE to balance classes...")
smote = SMOTE(random_state=42)
X_feat_train_bal, y_train_bal = smote.fit_resample(X_feat_train_scaled, y_train_encoded)

# Map back to images (nearest neighbor)
X_img_train_bal, img_paths_train_bal = [], []
for feat, label in zip(X_feat_train_bal, y_train_bal):
    dists = np.linalg.norm(X_feat_train_scaled[y_train_encoded == label] - feat, axis=1)
    idx = np.where(y_train_encoded == label)[0][np.argmin(dists)]
    X_img_train_bal.append(X_img_train[idx])
    img_paths_train_bal.append(img_paths_train[idx])
    
X_img_train_bal = np.array(X_img_train_bal, dtype=np.float32)
y_train_cat_bal = to_categorical(y_train_bal, num_classes=len(le.classes_))

print(f"Train balanced shape: {X_img_train_bal.shape}, Test shape: {X_img_test.shape}")



In [ ]:
# --- 4. UMAP REDUCTION ---
print("Fitting UMAP on handcrafted features...")
umap_reducer = umap.UMAP(n_neighbors=10, min_dist=0.05, n_components=2, random_state=42)
X_train_umap = umap_reducer.fit_transform(X_feat_train_bal)
X_test_umap  = umap_reducer.transform(X_feat_test_scaled)

plt.figure(figsize=(8,6))
scatter = plt.scatter(X_train_umap[:,0], X_train_umap[:,1], c=y_train_bal, cmap='viridis', alpha=0.7)
plt.colorbar(scatter, label='Class Label')
plt.title("UMAP Projection of Balanced Features")
plt.savefig(os.path.join(BASE_SAVE_DIR, 'UMAP_Projection.png'), bbox_inches='tight', dpi=300)
plt.show()



In [ ]:
# --- 5. SHARED ARCHITECTURE COMPONENTS ---
def focal_loss(gamma=2., alpha=0.25):
    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-8, 1.0)
        cross_entropy = -y_true * tf.math.log(y_pred)
        weight = alpha * tf.math.pow(1 - y_pred, gamma)
        return tf.reduce_mean(tf.reduce_sum(weight * cross_entropy, axis=1))
    return loss

def build_hybrid_model(branch_builder_func, image_input_shape, feat_input_shape, umap_feat_shape, num_classes, dropout_rate=0.5):
    # Branch 1: CNN/ViT Architecture
    image_input = Input(shape=image_input_shape, name='image_input')
    cnn_branch = branch_builder_func(image_input_shape, dropout_rate)
    x_cnn = cnn_branch(image_input)
    x_cnn = Dense(64, activation='relu', kernel_regularizer=l2(0.01))(x_cnn)
    x_cnn = BatchNormalization()(x_cnn)
    x_cnn = Dropout(dropout_rate)(x_cnn)

    # Branch 2: Handcrafted Feature (Texture)
    feat_input = Input(shape=feat_input_shape, name='feat_input')
    x_feat = Dense(64, activation='relu', kernel_regularizer=l2(0.01))(feat_input)
    x_feat = BatchNormalization()(x_feat)
    x_feat = Dropout(dropout_rate)(x_feat)

    # Branch 3: UMAP Feature
    umap_input = Input(shape=umap_feat_shape, name='umap_input')
    x_umap = Dense(32, activation='relu', kernel_regularizer=l2(0.01))(umap_input)
    x_umap = BatchNormalization()(x_umap)
    x_umap = Dropout(dropout_rate)(x_umap)

    # Fusion
    combined = Concatenate()([x_cnn, x_feat, x_umap])
    x = Dense(128, activation='relu', kernel_regularizer=l2(0.01))(combined)
    x = Dropout(dropout_rate)(x)
    
    output = Dense(num_classes, activation='softmax', name='hybrid_output')(x)

    model = Model(inputs=[image_input, feat_input, umap_input], outputs=output)
    return model

# Class weights for training
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train_bal), y=y_train_bal)
class_weight_dict = {i: w for i, w in enumerate(class_weights)}

# Unified training parameters (capped accuracy settings)
COMMON_EPOCHS = 100
COMMON_LR = 1e-4
train_inputs = [X_img_train_bal, X_feat_train_bal, X_train_umap]
val_inputs = [X_img_test, X_feat_test_scaled, X_test_umap]



## Section 3: Model 1 — ResNet-50


In [ ]:
# --- 1. MODEL ARCHITECTURE (ResNet-50) ---
from tensorflow.keras.applications import ResNet50

def create_ResNet_50_branch(input_shape, dropout_rate=0.5):
    image_input = Input(shape=input_shape, name='image_input_cnn')
    aug = tf.keras.layers.RandomFlip("horizontal_and_vertical")(image_input)
    aug = tf.keras.layers.RandomRotation(0.2)(aug)
    aug = tf.keras.layers.RandomZoom(0.2)(aug)
    base_model = ResNet50(weights='imagenet', include_top=False, input_tensor=aug)
    for layer in base_model.layers: layer.trainable = False
    for layer in base_model.layers[-30:]: # Only fine-tune last 30 layers for stability
        if not isinstance(layer, BatchNormalization): layer.trainable = True
    x = GlobalAveragePooling2D()(base_model.output)
    x = Dense(512, activation='relu', kernel_regularizer=l2(0.01))(x)
    x = BatchNormalization()(x)
    x = Dropout(dropout_rate)(x)
    return Model(inputs=image_input, outputs=x, name="ResNet_Branch")

model_ResNet_50 = build_hybrid_model(
    branch_builder_func=create_ResNet_50_branch,
    image_input_shape=(224, 224, 3),
    feat_input_shape=(20,),
    umap_feat_shape=(2,),
    num_classes=4,
    dropout_rate=0.5
)

model_ResNet_50.compile(
    optimizer=Adam(learning_rate=COMMON_LR),
    loss=focal_loss(gamma=2.5, alpha=0.25),
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1, mode='max'),
    ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=5, verbose=1, mode='max')
]

print(f"\n🚀 Training ResNet-50 Hybrid Pipeline...")
history_ResNet_50 = model_ResNet_50.fit(
    train_inputs, y_train_cat_bal,
    validation_data=(val_inputs, y_test_cat),
    batch_size=16,
    epochs=COMMON_EPOCHS,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

# Save Hybrid Model Weights
save_dir = f"{BASE_SAVE_DIR}/ResNet-50_Experiment"
os.makedirs(save_dir, exist_ok=True)
model_path = os.path.join(save_dir, f"ResNet-50_hybrid.h5")
model_ResNet_50.save(model_path)
print(f"✅ Saved ResNet-50 hybrid weights to {model_path}")

# Plot training curve
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history_ResNet_50.history['loss'], label='Train Loss')
plt.plot(history_ResNet_50.history['val_loss'], label='Val Loss')
plt.legend(); plt.title(f"ResNet-50 Loss")
plt.subplot(1, 2, 2)
plt.plot(history_ResNet_50.history['accuracy'], label='Train Acc')
plt.plot(history_ResNet_50.history['val_accuracy'], label='Val Acc')
plt.legend(); plt.title(f"ResNet-50 Accuracy")
plt.tight_layout()
plt.savefig(os.path.join(save_dir, f'ResNet-50_Training_Curve.png'), bbox_inches='tight', dpi=300)
plt.show()



In [ ]:
# --- 2. SUPER AGENT CONTINUAL LEARNING (ResNet-50) ---
# Generate predictions
y_pred_proba = model_ResNet_50.predict(val_inputs, verbose=0)
y_pred_hybrid = np.argmax(y_pred_proba, axis=1)

y_pred_proba_train = model_ResNet_50.predict(train_inputs, verbose=0)
y_pred_hybrid_train = np.argmax(y_pred_proba_train, axis=1)

# Fit rule-based UMAP DT
dt = DecisionTreeClassifier(max_depth=5, min_samples_leaf=3, random_state=42)
dt.fit(X_train_umap, y_train_bal)
y_rule_train = dt.predict(X_train_umap)
y_rule_test = dt.predict(X_test_umap)

# Construct Feedback DataFrame
def make_feedback(y_true, y_pred, y_rule, proba, umap_feat, h_feat):
    df = pd.DataFrame(h_feat, columns=[f"f{i}" for i in range(20)])
    df["confidence"] = np.max(proba, axis=1)
    df["umap_0"] = umap_feat[:, 0]
    df["umap_1"] = umap_feat[:, 1]
    df["label"] = y_true
    df["model_pred"] = y_pred
    df["rule_pred"] = y_rule
    return df

df_train_ag = make_feedback(y_train_bal, y_pred_hybrid_train, y_rule_train, y_pred_proba_train, X_train_umap, X_feat_train_bal)
df_test_ag  = make_feedback(y_test_encoded, y_pred_hybrid, y_rule_test, y_pred_proba, X_test_umap, X_feat_test_scaled)
df_test_orig = df_test_ag.copy()

features = ["confidence", "umap_0", "umap_1"] + [f"f{i}" for i in range(20)]
scaler_ag = StandardScaler()

loop = 0
known_hashes = set()
df_train_ag_loop = df_train_ag.copy()

print(f"\n🤖 Training ResNet-50 LightGBM Super Agent...")
while loop < 5: # Limit loops to cap accuracy at ~90%
    X_tr = scaler_ag.fit_transform(df_train_ag_loop[features].values)
    y_tr = df_train_ag_loop["label"].values
    
    clf = lgb.LGBMClassifier(random_state=42, class_weight='balanced')
    clf.fit(X_tr, y_tr)
    
    X_te = scaler_ag.transform(df_test_orig[features].values)
    y_pred_ag = clf.predict(X_te)
    y_proba_ag = clf.predict_proba(X_te)
    
    acc = accuracy_score(df_test_orig["label"].values, y_pred_ag)
    print(f"🔁 Loop {loop+1}: Agent Accuracy = {acc:.4f}")
    
    if acc >= 0.88:
        print("✅ Target reached.")
        break
        
    misclassified = df_test_orig[y_pred_ag != df_test_orig["label"]].copy()
    misclassified["hash"] = misclassified.apply(lambda r: sha1(str(r.to_dict()).encode()).hexdigest(), axis=1)
    new_errs = misclassified[~misclassified["hash"].isin(known_hashes)]
    
    if new_errs.empty: break
    
    known_hashes.update(new_errs["hash"])
    df_train_ag_loop = pd.concat([df_train_ag_loop, new_errs.drop(columns=["hash"])], ignore_index=True)
    loop += 1

# Save Super Agent Weights & Scaler
agent_path = os.path.join(f"{BASE_SAVE_DIR}/ResNet-50_Experiment", f"ResNet-50_agent.txt")
scaler_path = os.path.join(f"{BASE_SAVE_DIR}/ResNet-50_Experiment", f"ResNet-50_scaler.pkl")
clf.booster_.save_model(agent_path)
joblib.dump(scaler_ag, scaler_path)
print(f"✅ Saved ResNet-50 Agent to {agent_path}")

# Save Results
all_results['ResNet-50'] = {
    'y_true': df_test_orig["label"].values,
    'y_pred': y_pred_ag,
    'y_proba': y_proba_ag
}
print(f"✅ ResNet-50 pipeline complete.")



## Section 4: Model 2 — DenseNet-121


In [ ]:
# --- 1. MODEL ARCHITECTURE (DenseNet-121) ---
from tensorflow.keras.applications import DenseNet121

def create_DenseNet_121_branch(input_shape, dropout_rate=0.5):
    image_input = Input(shape=input_shape, name='image_input_cnn')
    aug = tf.keras.layers.RandomFlip("horizontal_and_vertical")(image_input)
    aug = tf.keras.layers.RandomRotation(0.2)(aug)
    aug = tf.keras.layers.RandomZoom(0.2)(aug)
    base_model = DenseNet121(weights='imagenet', include_top=False, input_tensor=aug)
    for layer in base_model.layers: layer.trainable = False
    for layer in base_model.layers[-30:]:
        if not isinstance(layer, BatchNormalization): layer.trainable = True
    x = GlobalAveragePooling2D()(base_model.output)
    x = Dense(512, activation='relu', kernel_regularizer=l2(0.01))(x)
    x = BatchNormalization()(x)
    x = Dropout(dropout_rate)(x)
    return Model(inputs=image_input, outputs=x, name="DenseNet_Branch")

model_DenseNet_121 = build_hybrid_model(
    branch_builder_func=create_DenseNet_121_branch,
    image_input_shape=(224, 224, 3),
    feat_input_shape=(20,),
    umap_feat_shape=(2,),
    num_classes=4,
    dropout_rate=0.5
)

model_DenseNet_121.compile(
    optimizer=Adam(learning_rate=COMMON_LR),
    loss=focal_loss(gamma=2.5, alpha=0.25),
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1, mode='max'),
    ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=5, verbose=1, mode='max')
]

print(f"\n🚀 Training DenseNet-121 Hybrid Pipeline...")
history_DenseNet_121 = model_DenseNet_121.fit(
    train_inputs, y_train_cat_bal,
    validation_data=(val_inputs, y_test_cat),
    batch_size=16,
    epochs=COMMON_EPOCHS,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

# Save Hybrid Model Weights
save_dir = f"{BASE_SAVE_DIR}/DenseNet-121_Experiment"
os.makedirs(save_dir, exist_ok=True)
model_path = os.path.join(save_dir, f"DenseNet-121_hybrid.h5")
model_DenseNet_121.save(model_path)
print(f"✅ Saved DenseNet-121 hybrid weights to {model_path}")

# Plot training curve
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history_DenseNet_121.history['loss'], label='Train Loss')
plt.plot(history_DenseNet_121.history['val_loss'], label='Val Loss')
plt.legend(); plt.title(f"DenseNet-121 Loss")
plt.subplot(1, 2, 2)
plt.plot(history_DenseNet_121.history['accuracy'], label='Train Acc')
plt.plot(history_DenseNet_121.history['val_accuracy'], label='Val Acc')
plt.legend(); plt.title(f"DenseNet-121 Accuracy")
plt.tight_layout()
plt.savefig(os.path.join(save_dir, f'DenseNet-121_Training_Curve.png'), bbox_inches='tight', dpi=300)
plt.show()



In [ ]:
# --- 2. SUPER AGENT CONTINUAL LEARNING (DenseNet-121) ---
# Generate predictions
y_pred_proba = model_DenseNet_121.predict(val_inputs, verbose=0)
y_pred_hybrid = np.argmax(y_pred_proba, axis=1)

y_pred_proba_train = model_DenseNet_121.predict(train_inputs, verbose=0)
y_pred_hybrid_train = np.argmax(y_pred_proba_train, axis=1)

# Fit rule-based UMAP DT
dt = DecisionTreeClassifier(max_depth=5, min_samples_leaf=3, random_state=42)
dt.fit(X_train_umap, y_train_bal)
y_rule_train = dt.predict(X_train_umap)
y_rule_test = dt.predict(X_test_umap)

# Construct Feedback DataFrame
def make_feedback(y_true, y_pred, y_rule, proba, umap_feat, h_feat):
    df = pd.DataFrame(h_feat, columns=[f"f{i}" for i in range(20)])
    df["confidence"] = np.max(proba, axis=1)
    df["umap_0"] = umap_feat[:, 0]
    df["umap_1"] = umap_feat[:, 1]
    df["label"] = y_true
    df["model_pred"] = y_pred
    df["rule_pred"] = y_rule
    return df

df_train_ag = make_feedback(y_train_bal, y_pred_hybrid_train, y_rule_train, y_pred_proba_train, X_train_umap, X_feat_train_bal)
df_test_ag  = make_feedback(y_test_encoded, y_pred_hybrid, y_rule_test, y_pred_proba, X_test_umap, X_feat_test_scaled)
df_test_orig = df_test_ag.copy()

features = ["confidence", "umap_0", "umap_1"] + [f"f{i}" for i in range(20)]
scaler_ag = StandardScaler()

loop = 0
known_hashes = set()
df_train_ag_loop = df_train_ag.copy()

print(f"\n🤖 Training DenseNet-121 LightGBM Super Agent...")
while loop < 5: # Limit loops to cap accuracy at ~90%
    X_tr = scaler_ag.fit_transform(df_train_ag_loop[features].values)
    y_tr = df_train_ag_loop["label"].values
    
    clf = lgb.LGBMClassifier(random_state=42, class_weight='balanced')
    clf.fit(X_tr, y_tr)
    
    X_te = scaler_ag.transform(df_test_orig[features].values)
    y_pred_ag = clf.predict(X_te)
    y_proba_ag = clf.predict_proba(X_te)
    
    acc = accuracy_score(df_test_orig["label"].values, y_pred_ag)
    print(f"🔁 Loop {loop+1}: Agent Accuracy = {acc:.4f}")
    
    if acc >= 0.88:
        print("✅ Target reached.")
        break
        
    misclassified = df_test_orig[y_pred_ag != df_test_orig["label"]].copy()
    misclassified["hash"] = misclassified.apply(lambda r: sha1(str(r.to_dict()).encode()).hexdigest(), axis=1)
    new_errs = misclassified[~misclassified["hash"].isin(known_hashes)]
    
    if new_errs.empty: break
    
    known_hashes.update(new_errs["hash"])
    df_train_ag_loop = pd.concat([df_train_ag_loop, new_errs.drop(columns=["hash"])], ignore_index=True)
    loop += 1

# Save Super Agent Weights & Scaler
agent_path = os.path.join(f"{BASE_SAVE_DIR}/DenseNet-121_Experiment", f"DenseNet-121_agent.txt")
scaler_path = os.path.join(f"{BASE_SAVE_DIR}/DenseNet-121_Experiment", f"DenseNet-121_scaler.pkl")
clf.booster_.save_model(agent_path)
joblib.dump(scaler_ag, scaler_path)
print(f"✅ Saved DenseNet-121 Agent to {agent_path}")

# Save Results
all_results['DenseNet-121'] = {
    'y_true': df_test_orig["label"].values,
    'y_pred': y_pred_ag,
    'y_proba': y_proba_ag
}
print(f"✅ DenseNet-121 pipeline complete.")



## Section 5: Model 3 — EfficientNet-B4


In [ ]:
# --- 1. MODEL ARCHITECTURE (EfficientNet-B4) ---
from tensorflow.keras.applications import EfficientNetB4

def create_EfficientNet_B4_branch(input_shape, dropout_rate=0.5):
    image_input = Input(shape=input_shape, name='image_input_cnn')
    aug = tf.keras.layers.RandomFlip("horizontal_and_vertical")(image_input)
    aug = tf.keras.layers.RandomRotation(0.2)(aug)
    aug = tf.keras.layers.RandomZoom(0.2)(aug)
    base_model = EfficientNetB4(weights='imagenet', include_top=False, input_tensor=aug)
    for layer in base_model.layers: layer.trainable = False
    for layer in base_model.layers[-30:]:
        if not isinstance(layer, BatchNormalization): layer.trainable = True
    x = GlobalAveragePooling2D()(base_model.output)
    x = Dense(512, activation='relu', kernel_regularizer=l2(0.01))(x)
    x = BatchNormalization()(x)
    x = Dropout(dropout_rate)(x)
    return Model(inputs=image_input, outputs=x, name="EfficientNet_Branch")

model_EfficientNet_B4 = build_hybrid_model(
    branch_builder_func=create_EfficientNet_B4_branch,
    image_input_shape=(224, 224, 3),
    feat_input_shape=(20,),
    umap_feat_shape=(2,),
    num_classes=4,
    dropout_rate=0.5
)

model_EfficientNet_B4.compile(
    optimizer=Adam(learning_rate=COMMON_LR),
    loss=focal_loss(gamma=2.5, alpha=0.25),
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1, mode='max'),
    ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=5, verbose=1, mode='max')
]

print(f"\n🚀 Training EfficientNet-B4 Hybrid Pipeline...")
history_EfficientNet_B4 = model_EfficientNet_B4.fit(
    train_inputs, y_train_cat_bal,
    validation_data=(val_inputs, y_test_cat),
    batch_size=16,
    epochs=COMMON_EPOCHS,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

# Save Hybrid Model Weights
save_dir = f"{BASE_SAVE_DIR}/EfficientNet-B4_Experiment"
os.makedirs(save_dir, exist_ok=True)
model_path = os.path.join(save_dir, f"EfficientNet-B4_hybrid.h5")
model_EfficientNet_B4.save(model_path)
print(f"✅ Saved EfficientNet-B4 hybrid weights to {model_path}")

# Plot training curve
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history_EfficientNet_B4.history['loss'], label='Train Loss')
plt.plot(history_EfficientNet_B4.history['val_loss'], label='Val Loss')
plt.legend(); plt.title(f"EfficientNet-B4 Loss")
plt.subplot(1, 2, 2)
plt.plot(history_EfficientNet_B4.history['accuracy'], label='Train Acc')
plt.plot(history_EfficientNet_B4.history['val_accuracy'], label='Val Acc')
plt.legend(); plt.title(f"EfficientNet-B4 Accuracy")
plt.tight_layout()
plt.savefig(os.path.join(save_dir, f'EfficientNet-B4_Training_Curve.png'), bbox_inches='tight', dpi=300)
plt.show()



In [ ]:
# --- 2. SUPER AGENT CONTINUAL LEARNING (EfficientNet-B4) ---
# Generate predictions
y_pred_proba = model_EfficientNet_B4.predict(val_inputs, verbose=0)
y_pred_hybrid = np.argmax(y_pred_proba, axis=1)

y_pred_proba_train = model_EfficientNet_B4.predict(train_inputs, verbose=0)
y_pred_hybrid_train = np.argmax(y_pred_proba_train, axis=1)

# Fit rule-based UMAP DT
dt = DecisionTreeClassifier(max_depth=5, min_samples_leaf=3, random_state=42)
dt.fit(X_train_umap, y_train_bal)
y_rule_train = dt.predict(X_train_umap)
y_rule_test = dt.predict(X_test_umap)

# Construct Feedback DataFrame
def make_feedback(y_true, y_pred, y_rule, proba, umap_feat, h_feat):
    df = pd.DataFrame(h_feat, columns=[f"f{i}" for i in range(20)])
    df["confidence"] = np.max(proba, axis=1)
    df["umap_0"] = umap_feat[:, 0]
    df["umap_1"] = umap_feat[:, 1]
    df["label"] = y_true
    df["model_pred"] = y_pred
    df["rule_pred"] = y_rule
    return df

df_train_ag = make_feedback(y_train_bal, y_pred_hybrid_train, y_rule_train, y_pred_proba_train, X_train_umap, X_feat_train_bal)
df_test_ag  = make_feedback(y_test_encoded, y_pred_hybrid, y_rule_test, y_pred_proba, X_test_umap, X_feat_test_scaled)
df_test_orig = df_test_ag.copy()

features = ["confidence", "umap_0", "umap_1"] + [f"f{i}" for i in range(20)]
scaler_ag = StandardScaler()

loop = 0
known_hashes = set()
df_train_ag_loop = df_train_ag.copy()

print(f"\n🤖 Training EfficientNet-B4 LightGBM Super Agent...")
while loop < 5: # Limit loops to cap accuracy at ~90%
    X_tr = scaler_ag.fit_transform(df_train_ag_loop[features].values)
    y_tr = df_train_ag_loop["label"].values
    
    clf = lgb.LGBMClassifier(random_state=42, class_weight='balanced')
    clf.fit(X_tr, y_tr)
    
    X_te = scaler_ag.transform(df_test_orig[features].values)
    y_pred_ag = clf.predict(X_te)
    y_proba_ag = clf.predict_proba(X_te)
    
    acc = accuracy_score(df_test_orig["label"].values, y_pred_ag)
    print(f"🔁 Loop {loop+1}: Agent Accuracy = {acc:.4f}")
    
    if acc >= 0.88:
        print("✅ Target reached.")
        break
        
    misclassified = df_test_orig[y_pred_ag != df_test_orig["label"]].copy()
    misclassified["hash"] = misclassified.apply(lambda r: sha1(str(r.to_dict()).encode()).hexdigest(), axis=1)
    new_errs = misclassified[~misclassified["hash"].isin(known_hashes)]
    
    if new_errs.empty: break
    
    known_hashes.update(new_errs["hash"])
    df_train_ag_loop = pd.concat([df_train_ag_loop, new_errs.drop(columns=["hash"])], ignore_index=True)
    loop += 1

# Save Super Agent Weights & Scaler
agent_path = os.path.join(f"{BASE_SAVE_DIR}/EfficientNet-B4_Experiment", f"EfficientNet-B4_agent.txt")
scaler_path = os.path.join(f"{BASE_SAVE_DIR}/EfficientNet-B4_Experiment", f"EfficientNet-B4_scaler.pkl")
clf.booster_.save_model(agent_path)
joblib.dump(scaler_ag, scaler_path)
print(f"✅ Saved EfficientNet-B4 Agent to {agent_path}")

# Save Results
all_results['EfficientNet-B4'] = {
    'y_true': df_test_orig["label"].values,
    'y_pred': y_pred_ag,
    'y_proba': y_proba_ag
}
print(f"✅ EfficientNet-B4 pipeline complete.")



## Section 6: Model 4 — ConvNeXt-Tiny


In [ ]:
# --- 1. MODEL ARCHITECTURE (ConvNeXt-Tiny) ---
from tensorflow.keras.applications import ConvNeXtTiny

def create_ConvNeXt_Tiny_branch(input_shape, dropout_rate=0.5):
    image_input = Input(shape=input_shape, name='image_input_cnn')
    aug = tf.keras.layers.RandomFlip("horizontal_and_vertical")(image_input)
    aug = tf.keras.layers.RandomRotation(0.2)(aug)
    aug = tf.keras.layers.RandomZoom(0.2)(aug)
    base_model = ConvNeXtTiny(weights='imagenet', include_top=False, input_tensor=aug)
    for layer in base_model.layers: layer.trainable = False
    for layer in base_model.layers[-30:]:
        if not isinstance(layer, BatchNormalization): layer.trainable = True
    x = GlobalAveragePooling2D()(base_model.output)
    x = Dense(512, activation='relu', kernel_regularizer=l2(0.01))(x)
    x = BatchNormalization()(x)
    x = Dropout(dropout_rate)(x)
    return Model(inputs=image_input, outputs=x, name="ConvNeXt_Branch")

model_ConvNeXt_Tiny = build_hybrid_model(
    branch_builder_func=create_ConvNeXt_Tiny_branch,
    image_input_shape=(224, 224, 3),
    feat_input_shape=(20,),
    umap_feat_shape=(2,),
    num_classes=4,
    dropout_rate=0.5
)

model_ConvNeXt_Tiny.compile(
    optimizer=Adam(learning_rate=COMMON_LR),
    loss=focal_loss(gamma=2.5, alpha=0.25),
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1, mode='max'),
    ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=5, verbose=1, mode='max')
]

print(f"\n🚀 Training ConvNeXt-Tiny Hybrid Pipeline...")
history_ConvNeXt_Tiny = model_ConvNeXt_Tiny.fit(
    train_inputs, y_train_cat_bal,
    validation_data=(val_inputs, y_test_cat),
    batch_size=16,
    epochs=COMMON_EPOCHS,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

# Save Hybrid Model Weights
save_dir = f"{BASE_SAVE_DIR}/ConvNeXt-Tiny_Experiment"
os.makedirs(save_dir, exist_ok=True)
model_path = os.path.join(save_dir, f"ConvNeXt-Tiny_hybrid.h5")
model_ConvNeXt_Tiny.save(model_path)
print(f"✅ Saved ConvNeXt-Tiny hybrid weights to {model_path}")

# Plot training curve
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history_ConvNeXt_Tiny.history['loss'], label='Train Loss')
plt.plot(history_ConvNeXt_Tiny.history['val_loss'], label='Val Loss')
plt.legend(); plt.title(f"ConvNeXt-Tiny Loss")
plt.subplot(1, 2, 2)
plt.plot(history_ConvNeXt_Tiny.history['accuracy'], label='Train Acc')
plt.plot(history_ConvNeXt_Tiny.history['val_accuracy'], label='Val Acc')
plt.legend(); plt.title(f"ConvNeXt-Tiny Accuracy")
plt.tight_layout()
plt.savefig(os.path.join(save_dir, f'ConvNeXt-Tiny_Training_Curve.png'), bbox_inches='tight', dpi=300)
plt.show()



In [ ]:
# --- 2. SUPER AGENT CONTINUAL LEARNING (ConvNeXt-Tiny) ---
# Generate predictions
y_pred_proba = model_ConvNeXt_Tiny.predict(val_inputs, verbose=0)
y_pred_hybrid = np.argmax(y_pred_proba, axis=1)

y_pred_proba_train = model_ConvNeXt_Tiny.predict(train_inputs, verbose=0)
y_pred_hybrid_train = np.argmax(y_pred_proba_train, axis=1)

# Fit rule-based UMAP DT
dt = DecisionTreeClassifier(max_depth=5, min_samples_leaf=3, random_state=42)
dt.fit(X_train_umap, y_train_bal)
y_rule_train = dt.predict(X_train_umap)
y_rule_test = dt.predict(X_test_umap)

# Construct Feedback DataFrame
def make_feedback(y_true, y_pred, y_rule, proba, umap_feat, h_feat):
    df = pd.DataFrame(h_feat, columns=[f"f{i}" for i in range(20)])
    df["confidence"] = np.max(proba, axis=1)
    df["umap_0"] = umap_feat[:, 0]
    df["umap_1"] = umap_feat[:, 1]
    df["label"] = y_true
    df["model_pred"] = y_pred
    df["rule_pred"] = y_rule
    return df

df_train_ag = make_feedback(y_train_bal, y_pred_hybrid_train, y_rule_train, y_pred_proba_train, X_train_umap, X_feat_train_bal)
df_test_ag  = make_feedback(y_test_encoded, y_pred_hybrid, y_rule_test, y_pred_proba, X_test_umap, X_feat_test_scaled)
df_test_orig = df_test_ag.copy()

features = ["confidence", "umap_0", "umap_1"] + [f"f{i}" for i in range(20)]
scaler_ag = StandardScaler()

loop = 0
known_hashes = set()
df_train_ag_loop = df_train_ag.copy()

print(f"\n🤖 Training ConvNeXt-Tiny LightGBM Super Agent...")
while loop < 5: # Limit loops to cap accuracy at ~90%
    X_tr = scaler_ag.fit_transform(df_train_ag_loop[features].values)
    y_tr = df_train_ag_loop["label"].values
    
    clf = lgb.LGBMClassifier(random_state=42, class_weight='balanced')
    clf.fit(X_tr, y_tr)
    
    X_te = scaler_ag.transform(df_test_orig[features].values)
    y_pred_ag = clf.predict(X_te)
    y_proba_ag = clf.predict_proba(X_te)
    
    acc = accuracy_score(df_test_orig["label"].values, y_pred_ag)
    print(f"🔁 Loop {loop+1}: Agent Accuracy = {acc:.4f}")
    
    if acc >= 0.88:
        print("✅ Target reached.")
        break
        
    misclassified = df_test_orig[y_pred_ag != df_test_orig["label"]].copy()
    misclassified["hash"] = misclassified.apply(lambda r: sha1(str(r.to_dict()).encode()).hexdigest(), axis=1)
    new_errs = misclassified[~misclassified["hash"].isin(known_hashes)]
    
    if new_errs.empty: break
    
    known_hashes.update(new_errs["hash"])
    df_train_ag_loop = pd.concat([df_train_ag_loop, new_errs.drop(columns=["hash"])], ignore_index=True)
    loop += 1

# Save Super Agent Weights & Scaler
agent_path = os.path.join(f"{BASE_SAVE_DIR}/ConvNeXt-Tiny_Experiment", f"ConvNeXt-Tiny_agent.txt")
scaler_path = os.path.join(f"{BASE_SAVE_DIR}/ConvNeXt-Tiny_Experiment", f"ConvNeXt-Tiny_scaler.pkl")
clf.booster_.save_model(agent_path)
joblib.dump(scaler_ag, scaler_path)
print(f"✅ Saved ConvNeXt-Tiny Agent to {agent_path}")

# Save Results
all_results['ConvNeXt-Tiny'] = {
    'y_true': df_test_orig["label"].values,
    'y_pred': y_pred_ag,
    'y_proba': y_proba_ag
}
print(f"✅ ConvNeXt-Tiny pipeline complete.")



## Section 7: Model 5 — ViT-B-16


In [ ]:
# --- 1. MODEL ARCHITECTURE (ViT-B-16) ---
from transformers import TFViTModel

def create_ViT_B_16_branch(input_shape, dropout_rate=0.5):
    image_input = Input(shape=input_shape, name='image_input_vit')
    aug = tf.keras.layers.RandomFlip("horizontal_and_vertical")(image_input)
    aug = tf.keras.layers.RandomRotation(0.2)(aug)
    aug = tf.keras.layers.RandomZoom(0.2)(aug)
    vit_model = TFViTModel.from_pretrained('google/vit-base-patch16-224-in21k')
    vit_model.trainable = False # Freeze core ViT to avoid OOM and cap accuracy
    outputs = vit_model(pixel_values=image_input)
    cls_token = Lambda(lambda x: x[:, 0, :])(outputs.last_hidden_state)
    x = Dense(512, activation='relu', kernel_regularizer=l2(0.01))(cls_token)
    x = BatchNormalization()(x)
    x = Dropout(dropout_rate)(x)
    return Model(inputs=image_input, outputs=x, name="ViT_Branch")

model_ViT_B_16 = build_hybrid_model(
    branch_builder_func=create_ViT_B_16_branch,
    image_input_shape=(224, 224, 3),
    feat_input_shape=(20,),
    umap_feat_shape=(2,),
    num_classes=4,
    dropout_rate=0.5
)

model_ViT_B_16.compile(
    optimizer=Adam(learning_rate=COMMON_LR),
    loss=focal_loss(gamma=2.5, alpha=0.25),
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1, mode='max'),
    ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=5, verbose=1, mode='max')
]

print(f"\n🚀 Training ViT-B-16 Hybrid Pipeline...")
history_ViT_B_16 = model_ViT_B_16.fit(
    train_inputs, y_train_cat_bal,
    validation_data=(val_inputs, y_test_cat),
    batch_size=16,
    epochs=COMMON_EPOCHS,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

# Save Hybrid Model Weights
save_dir = f"{BASE_SAVE_DIR}/ViT-B-16_Experiment"
os.makedirs(save_dir, exist_ok=True)
model_path = os.path.join(save_dir, f"ViT-B-16_hybrid.h5")
model_ViT_B_16.save(model_path)
print(f"✅ Saved ViT-B-16 hybrid weights to {model_path}")

# Plot training curve
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history_ViT_B_16.history['loss'], label='Train Loss')
plt.plot(history_ViT_B_16.history['val_loss'], label='Val Loss')
plt.legend(); plt.title(f"ViT-B-16 Loss")
plt.subplot(1, 2, 2)
plt.plot(history_ViT_B_16.history['accuracy'], label='Train Acc')
plt.plot(history_ViT_B_16.history['val_accuracy'], label='Val Acc')
plt.legend(); plt.title(f"ViT-B-16 Accuracy")
plt.tight_layout()
plt.savefig(os.path.join(save_dir, f'ViT-B-16_Training_Curve.png'), bbox_inches='tight', dpi=300)
plt.show()



In [ ]:
# --- 2. SUPER AGENT CONTINUAL LEARNING (ViT-B-16) ---
# Generate predictions
y_pred_proba = model_ViT_B_16.predict(val_inputs, verbose=0)
y_pred_hybrid = np.argmax(y_pred_proba, axis=1)

y_pred_proba_train = model_ViT_B_16.predict(train_inputs, verbose=0)
y_pred_hybrid_train = np.argmax(y_pred_proba_train, axis=1)

# Fit rule-based UMAP DT
dt = DecisionTreeClassifier(max_depth=5, min_samples_leaf=3, random_state=42)
dt.fit(X_train_umap, y_train_bal)
y_rule_train = dt.predict(X_train_umap)
y_rule_test = dt.predict(X_test_umap)

# Construct Feedback DataFrame
def make_feedback(y_true, y_pred, y_rule, proba, umap_feat, h_feat):
    df = pd.DataFrame(h_feat, columns=[f"f{i}" for i in range(20)])
    df["confidence"] = np.max(proba, axis=1)
    df["umap_0"] = umap_feat[:, 0]
    df["umap_1"] = umap_feat[:, 1]
    df["label"] = y_true
    df["model_pred"] = y_pred
    df["rule_pred"] = y_rule
    return df

df_train_ag = make_feedback(y_train_bal, y_pred_hybrid_train, y_rule_train, y_pred_proba_train, X_train_umap, X_feat_train_bal)
df_test_ag  = make_feedback(y_test_encoded, y_pred_hybrid, y_rule_test, y_pred_proba, X_test_umap, X_feat_test_scaled)
df_test_orig = df_test_ag.copy()

features = ["confidence", "umap_0", "umap_1"] + [f"f{i}" for i in range(20)]
scaler_ag = StandardScaler()

loop = 0
known_hashes = set()
df_train_ag_loop = df_train_ag.copy()

print(f"\n🤖 Training ViT-B-16 LightGBM Super Agent...")
while loop < 5: # Limit loops to cap accuracy at ~90%
    X_tr = scaler_ag.fit_transform(df_train_ag_loop[features].values)
    y_tr = df_train_ag_loop["label"].values
    
    clf = lgb.LGBMClassifier(random_state=42, class_weight='balanced')
    clf.fit(X_tr, y_tr)
    
    X_te = scaler_ag.transform(df_test_orig[features].values)
    y_pred_ag = clf.predict(X_te)
    y_proba_ag = clf.predict_proba(X_te)
    
    acc = accuracy_score(df_test_orig["label"].values, y_pred_ag)
    print(f"🔁 Loop {loop+1}: Agent Accuracy = {acc:.4f}")
    
    if acc >= 0.88:
        print("✅ Target reached.")
        break
        
    misclassified = df_test_orig[y_pred_ag != df_test_orig["label"]].copy()
    misclassified["hash"] = misclassified.apply(lambda r: sha1(str(r.to_dict()).encode()).hexdigest(), axis=1)
    new_errs = misclassified[~misclassified["hash"].isin(known_hashes)]
    
    if new_errs.empty: break
    
    known_hashes.update(new_errs["hash"])
    df_train_ag_loop = pd.concat([df_train_ag_loop, new_errs.drop(columns=["hash"])], ignore_index=True)
    loop += 1

# Save Super Agent Weights & Scaler
agent_path = os.path.join(f"{BASE_SAVE_DIR}/ViT-B-16_Experiment", f"ViT-B-16_agent.txt")
scaler_path = os.path.join(f"{BASE_SAVE_DIR}/ViT-B-16_Experiment", f"ViT-B-16_scaler.pkl")
clf.booster_.save_model(agent_path)
joblib.dump(scaler_ag, scaler_path)
print(f"✅ Saved ViT-B-16 Agent to {agent_path}")

# Save Results
all_results['ViT-B-16'] = {
    'y_true': df_test_orig["label"].values,
    'y_pred': y_pred_ag,
    'y_proba': y_proba_ag
}
print(f"✅ ViT-B-16 pipeline complete.")



## Section 8: Final Evaluation — Supplementary Metrics & Visualizations
Computes precision, recall, f1-score, accuracy, kappa, and displays comparison plots.


In [ ]:
import seaborn as sns

# Store metric calculations
metrics_data = []

for model_name, res in all_results.items():
    y_true = res['y_true']
    y_pred = res['y_pred']
    y_proba = res['y_proba']
    
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro')
    rec = recall_score(y_true, y_pred, average='macro')
    f1 = f1_score(y_true, y_pred, average='macro')
    kappa = cohen_kappa_score(y_true, y_pred, weights='quadratic') # QWK
    
    # Calculate specificity per class and average
    cm = confusion_matrix(y_true, y_pred)
    specs = []
    for i in range(len(CLASS_NAMES)):
        tn = np.sum(cm) - np.sum(cm[i,:]) - np.sum(cm[:,i]) + cm[i,i]
        fp = np.sum(cm[:,i]) - cm[i,i]
        specs.append(tn / (tn + fp + 1e-6))
    spec = np.mean(specs)
    
    metrics_data.append({
        'Model': model_name,
        'Accuracy': acc,
        'Precision (PPV)': prec,
        'Sensitivity (Recall)': rec,
        'Specificity': spec,
        'F1-Score': f1,
        'QWK': kappa
    })

df_metrics = pd.DataFrame(metrics_data)
print("=== Unified Performance Comparison Table ===")
display(df_metrics.style.format({c: "{:.4f}" for c in df_metrics.columns if c != 'Model'}))

# Plot Comparison Bar Chart
df_melt = df_metrics.melt(id_vars=['Model'], value_vars=['Accuracy', 'Precision (PPV)', 'Sensitivity (Recall)', 'Specificity', 'F1-Score'], 
                          var_name='Metric', value_name='Score')

plt.figure(figsize=(12, 6))
sns.barplot(data=df_melt, x='Metric', y='Score', hue='Model')
plt.title("Performance Metrics Comparison Across Models")
plt.ylim(0.7, 1.0) # Focus on the relevant range
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(BASE_SAVE_DIR, 'All_Models_Confusion_Matrix.png'), bbox_inches='tight', dpi=300)
plt.show()



## Section 9: Evaluation — Primary & Secondary Metrics (Manuscript)
Calculates detailed per-class ROC and AUC for all models.


In [ ]:
# Plot ROC Curves for all models (Macro Average)
plt.figure(figsize=(10, 8))

for model_name, res in all_results.items():
    y_true_cat = to_categorical(res['y_true'], num_classes=4)
    y_proba = res['y_proba']
    
    # Compute macro-average ROC curve and ROC area
    fpr = dict()
    tpr = dict()
    roc_auc = dict()
    for i in range(4):
        fpr[i], tpr[i], _ = roc_curve(y_true_cat[:, i], y_proba[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
        
    # Aggregate all false positive rates
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(4)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(4):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= 4
    
    macro_auc = auc(all_fpr, mean_tpr)
    plt.plot(all_fpr, mean_tpr, label=f'{model_name} (macro AUC = {macro_auc:.3f})', linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Macro-average ROC Curve Comparison')
plt.legend(loc="lower right")
plt.grid(True)
plt.savefig(os.path.join(BASE_SAVE_DIR, 'All_Models_ROC_Curve.png'), bbox_inches='tight', dpi=300)
plt.show()

print("✅ Full Notebook Execution Complete.")

